In [3]:
import pandas as pd
import numpy as np

# ===== 경로 =====
PRICE_PATH = "adj close exclude non trading day.csv"      # 일별 '조정종가' 와이드 (index=date, columns=tickers)
DV_PATH    = "trading value daily real 10.csv"   # 일별 '거래대금' 와이드 (index=date, columns=tickers)

MIN_DAYS_PER_MONTH = 10                  # 월 평균에 포함할 최소 거래일
OUT_DAY   = "illiq_daily9.csv"
OUT_MONTH = "illiq_monthly9.csv"

# ===== 유틸 =====
def read_wide(path):
    df = pd.read_csv(path, index_col=0)
    if not np.issubdtype(df.index.dtype, np.datetime64):
        df.index = pd.to_datetime(df.index)
    df.columns = df.columns.map(str)
    return df.sort_index()

# 1) 로드
price = read_wide(PRICE_PATH)
dval  = read_wide(DV_PATH) 




# 2) 칼럼
common = price.columns.intersection(dval.columns)
price  = price[common]
dval   = dval[common]

# 3) 전처리
price = price.replace([np.inf, -np.inf], np.nan)
price[price <= 0] = np.nan

dval  = dval.replace([np.inf, -np.inf], np.nan)
dval  = dval.replace(0, np.nan)  # 분모 0 방지

# 4) 일별 수익률 (단순수익률)
ret_d = price.pct_change()
ret_d = ret_d.replace([np.inf, -np.inf], np.nan)

# 5) 일별 ILLIQ = |r| / 거래대금
illiq_d = ret_d.abs() / dval
illiq_d = illiq_d.replace([0, np.inf, -np.inf], np.nan)

# (선택) 윈저라이즈로 극단값 컷하고 싶으면 주석 해제
def winsorize(df, lower=0.005, upper=0.995):
    ql = df.quantile(lower)
    qh = df.quantile(upper)
    return df.clip(lower=ql, upper=qh, axis=1)
illiq_d = winsorize(illiq_d)

# 6) 월별 평균 (최신 pandas 방식)
illiq_m = illiq_d.resample("M").mean()
cnt_m   = illiq_d.resample("M").count()
illiq_m[cnt_m < MIN_DAYS_PER_MONTH] = np.nan

# 7) 저장
illiq_d.to_csv(OUT_DAY, float_format="%.12g")
illiq_m.to_csv(OUT_MONTH, float_format="%.12g")

print("[OK] daily ILLIQ:", illiq_d.shape, "->", OUT_DAY)
print("[OK] monthly ILLIQ:", illiq_m.shape, "->", OUT_MONTH)


/var/folders/qj/vy5ztdf564jg38f9lktwgj6m0000gn/T/ipykernel_2435/3893840994.py:40: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ret_d = price.pct_change()
/var/folders/qj/vy5ztdf564jg38f9lktwgj6m0000gn/T/ipykernel_2435/3893840994.py:55: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  illiq_m = illiq_d.resample("M").mean()
/var/folders/qj/vy5ztdf564jg38f9lktwgj6m0000gn/T/ipykernel_2435/3893840994.py:56: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  cnt_m   = illiq_d.resample("M").count()


[OK] daily ILLIQ: (16741, 3874) -> illiq_daily9.csv
[OK] monthly ILLIQ: (550, 3874) -> illiq_monthly9.csv


In [4]:
import pandas as pd
import numpy as np

# 일별 ILLIQ 데이터 (index=date, columns=tickers)
illiq_d = pd.read_csv("illiq_monthly9.csv", index_col=0, parse_dates=True)


# 1️⃣ 이동평균 기간 설정 (예: 60일 = 약 3개월 추세)
window_1 = 1

window_3 = 3

window_6 = 6

window_12 = 12


# 2️⃣ 영구적 성분 (장기 추세)
illiq_perm_1 = illiq_d.rolling(window_1, min_periods=1).mean()
illiq_perm_3 = illiq_d.rolling(window_3, min_periods=3).mean()
illiq_perm_6 = illiq_d.rolling(window_6, min_periods=6).mean()
illiq_perm_12 = illiq_d.rolling(window_12, min_periods=12).mean()


# 4️⃣ 확인
print(illiq_perm_1.tail())
print(illiq_perm_3.tail())
print(illiq_perm_6.tail())
print(illiq_perm_12.tail())


# (선택) 월별 집계로 변환하고 싶다면
illiq_perm_m_1 = illiq_perm_1.resample("M").mean()
illiq_perm_m_3 = illiq_perm_3.resample("M").mean()
illiq_perm_m_6 = illiq_perm_6.resample("M").mean()
illiq_perm_m_12 = illiq_perm_12.resample("M").mean()



illiq_perm_m_1.to_csv("illiq_perm_monthly_1.csv")
illiq_perm_m_3.to_csv("illiq_perm_monthly_3.csv")
illiq_perm_m_6.to_csv("illiq_perm_monthly_6.csv")
illiq_perm_m_12.to_csv("illiq_perm_monthly_12.csv")

                    삼성전자        SK하이닉스      LG에너지솔루션      삼성바이오로직스  \
Date                                                                 
2025-06-30  1.465409e-14  2.515607e-14  2.340371e-13  1.108395e-13   
2025-07-31  1.140978e-14  1.668374e-14  1.815970e-13  1.758473e-13   
2025-08-31  1.113212e-14  3.227493e-14  2.044413e-13  1.900105e-13   
2025-09-30  1.282663e-14  2.812708e-14  2.026870e-13  1.142733e-13   
2025-10-31  9.104877e-15  1.913472e-14  1.582529e-13  1.210046e-13   

                     현대차       두산에너빌리티       HD현대중공업     한화에어로스페이스  \
Date                                                                 
2025-06-30  1.057737e-13  5.140020e-14  1.952711e-13  1.069392e-13   
2025-07-31  9.245045e-14  6.141018e-14  2.532420e-13  1.011141e-13   
2025-08-31  7.142087e-14  5.792702e-14  1.931696e-13  1.120640e-13   
2025-09-30  6.883461e-14  6.198123e-14  1.653247e-13  1.016720e-13   
2025-10-31  7.868541e-14  5.151671e-14  1.696910e-13  1.316341e-13   

                  

/var/folders/qj/vy5ztdf564jg38f9lktwgj6m0000gn/T/ipykernel_2435/708190330.py:33: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  illiq_perm_m_1 = illiq_perm_1.resample("M").mean()
/var/folders/qj/vy5ztdf564jg38f9lktwgj6m0000gn/T/ipykernel_2435/708190330.py:34: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  illiq_perm_m_3 = illiq_perm_3.resample("M").mean()
/var/folders/qj/vy5ztdf564jg38f9lktwgj6m0000gn/T/ipykernel_2435/708190330.py:35: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  illiq_perm_m_6 = illiq_perm_6.resample("M").mean()
/var/folders/qj/vy5ztdf564jg38f9lktwgj6m0000gn/T/ipykernel_2435/708190330.py:36: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  illiq_perm_m_12 = illiq_perm_12.resample("M").mean()


In [9]:
import pandas as pd
import numpy as np

def apply_rebalancing_wide(df_signals, rebal_freq_months):
    """
    롤링 평균 '신호' DataFrame 전체에 대해 
    리밸런싱 '홀딩' 로직을 적용합니다.
    (loc를 사용한 최종 수정 버전)

    Args:
        df_signals (pd.DataFrame): 원본 롤링 평균 데이터. 
                                   (index=날짜, columns=개별 주식)
        rebal_freq_months (int): 리밸런싱 주기 (예: 3, 6, 12)

    Returns:
        pd.DataFrame: 리밸런싱 홀딩 로직이 적용된 DataFrame
    """
    
    # 1. 리밸런싱 월인지 여부를 boolean 배열로 생성
    is_rebal_month = (df_signals.index.month % rebal_freq_months == 0)
    
    # 2. 리밸런싱이 일어나는 실제 날짜(인덱스)를 가져옴
    rebal_indices = df_signals.index[is_rebal_month]
    
    # 3. 원본과 동일한 모양이지만, 모두 NaN으로 채워진 임시 DataFrame 생성
    df_temp_signal = pd.DataFrame(np.nan, 
                                  index=df_signals.index, 
                                  columns=df_signals.columns)
    
    # 4. loc를 사용해 리밸런싱 날짜(rebal_indices)에만
    #    원본 신호(df_signals) 데이터를 복사해 넣음
    df_temp_signal.loc[rebal_indices] = df_signals.loc[rebal_indices]
    
    # 5. ffill()을 DataFrame 전체에 적용하여 NaN을 채움 (홀딩 효과)
    df_rebalanced = df_temp_signal.ffill()
    
    return df_rebalanced

# --- 1. 원본 데이터 로드 (가정) ---
#
# (이 부분에서 illiq_perm_m_3, illiq_perm_m_6, illiq_perm_m_12를
#  csv나 excel 파일 등에서 불러왔다고 가정합니다.)
#
# 예시:
# illiq_perm_m_3 = pd.read_pickle("illiq_perm_m_3.pkl")
# illiq_perm_m_6 = pd.read_pickle("illiq_perm_m_6.pkl")
# illiq_perm_m_12 = pd.read_pickle("illiq_perm_m_12.pkl")
#
# (만약 이 데이터가 이미 메모리에 로드되어 있다면, 이 단계는 필요 없습니다)


# --- 2. 요청한 6가지 리밸런싱 조합 생성 ---
# (원본 데이터프레임이 존재한다고 가정하고 바로 함수 호출)
print("리밸런싱 로직 적용 및 CSV 파일 저장 시작...")

# 1. 3개월 롤링 -> 3개월 리밸런싱
df_sig3_rebal3 = apply_rebalancing_wide(illiq_perm_m_3, 3)
df_sig3_rebal3.to_csv("df_sig3_rebal3.csv")

# 2. 6개월 롤링 -> 3개월 리밸런싱
df_sig6_rebal3 = apply_rebalancing_wide(illiq_perm_m_6, 3)
df_sig6_rebal3.to_csv("df_sig6_rebal3.csv")

# 3. 6개월 롤링 -> 6개월 리밸런싱
df_sig6_rebal6 = apply_rebalancing_wide(illiq_perm_m_6, 6)
df_sig6_rebal6.to_csv("df_sig6_rebal6.csv")

# 4. 12개월 롤링 -> 3개월 리밸런싱
df_sig12_rebal3 = apply_rebalancing_wide(illiq_perm_m_12, 3)
df_sig12_rebal3.to_csv("df_sig12_rebal3.csv")

# 5. 12개월 롤링 -> 6개월 리밸런싱
df_sig12_rebal6 = apply_rebalancing_wide(illiq_perm_m_12, 6)
df_sig12_rebal6.to_csv("df_sig12_rebal6.csv")

# 6. 12개월 롤링 -> 12개월 리밸런싱
df_sig12_rebal12 = apply_rebalancing_wide(illiq_perm_m_12, 12)
df_sig12_rebal12.to_csv("df_sig12_rebal12.csv")

print("모든 6개 파일 저장이 완료되었습니다.")

# --- 3. 결과 확인 (예시) ---
# 생성된 6개의 DataFrame 중 하나를 확인해봅니다.
print("\n--- 6개월 신호 + 3개월 리밸런싱 적용 (df_sig6_rebal3) 결과 (앞 7줄) ---")
print(df_sig6_rebal3.head(7))

print("\n--- 12개월 신호 + 12개월 리밸런싱 적용 (df_sig12_rebal12) 결과 (앞 13줄) ---")
print(df_sig12_rebal12.tail(13))

리밸런싱 로직 적용 및 CSV 파일 저장 시작...
모든 6개 파일 저장이 완료되었습니다.

--- 6개월 신호 + 3개월 리밸런싱 적용 (df_sig6_rebal3) 결과 (앞 7줄) ---
                    삼성전자  SK하이닉스  LG에너지솔루션  삼성바이오로직스           현대차  두산에너빌리티  \
Date                                                                          
1980-01-31           NaN     NaN       NaN       NaN           NaN      NaN   
1980-02-29           NaN     NaN       NaN       NaN           NaN      NaN   
1980-03-31           NaN     NaN       NaN       NaN           NaN      NaN   
1980-04-30           NaN     NaN       NaN       NaN           NaN      NaN   
1980-05-31           NaN     NaN       NaN       NaN           NaN      NaN   
1980-06-30  1.719178e-10     NaN       NaN       NaN  4.691551e-10      NaN   
1980-07-31  1.719178e-10     NaN       NaN       NaN  4.691551e-10      NaN   

            HD현대중공업  한화에어로스페이스            기아  KB금융  ...  웨이포트  성융광전투자  완리  \
Date                                                ...                     
1980-01-31      NaN       